In [1]:
import numpy as np
import os
import xarray as xr
import pandas as pd
import pickle
import glob

# --- Paths ---

home_dir = os.path.expanduser("~")

base_path = '/data/rong4/Data/ERA5/3hourly/quvw_US'

# Folder to save all profiles

# save_folder = os.path.join(home_dir, "advection_profiles_single_gridpoint")

# os.makedirs(save_folder, exist_ok=True)

years = [str(year) for year in range(2001, 2023)]

def get_files(folder, component):
    
    files = glob.glob(os.path.join(base_path, folder, f"era5.{component}.*.nc"))
    
    return [f for f in files if any(year in f for year in years)]

# --- Load Yizhou ERA5 files ---

u_files = get_files('u_component_of_wind', 'u_component_of_wind')

v_files = get_files('v_component_of_wind', 'v_component_of_wind')

q_files = get_files('specific_humidity', 'specific_humidity')

uvq_yz = xr.open_mfdataset(u_files + v_files + q_files,
                         
                        combine='by_coords', chunks={'time': 24})

# Filter by domain and warm season

uvq_yz = uvq_yz.sel(latitude=slice(39.25, 29.75), longitude=slice(-105.25, -94.75))

uvq_yz['time'] = uvq_yz['time'] - pd.Timedelta(hours=6)

uvq_yz = uvq_yz.where(
    
    (uvq_yz.time.dt.month.isin([5, 6, 7, 8, 9])) &
    
    (uvq_yz.time.dt.year.isin(range(2001, 2023))),
    
    drop=True
)

# Load my ERA5 files, 2022-2025

def list_nc_files(path, contains=None):
    
    path = os.path.expanduser(path)
    
    files = sorted(glob.glob(os.path.join(path, "*.nc")))
    
    if contains is not None:
        
        files = [f for f in files if contains in os.path.basename(f)]
    
    return files

# ------------------------------------------------------------
# 1) build uvq_new from your 2022–2025 paths
# ------------------------------------------------------------

u_2022_2025_path = '~/data/era5sgp/era5_ml_u_full_sgp_2022_2025'

v_2022_2025_path = '~/data/era5sgp/era5_ml_v_full_sgp_2022_2025'

q_2022_2025_path = '~/data/era5sgp/era5_ml_q_full_sgp_2022_2025'

u_files_new = list_nc_files(u_2022_2025_path)

v_files_new = list_nc_files(v_2022_2025_path)

q_files_new = list_nc_files(q_2022_2025_path)

uvq_new = xr.open_mfdataset(
    
    u_files_new + v_files_new + q_files_new,
    
    combine="by_coords",
    
    chunks={"time": 24})

uvq_new = uvq_new.sel(latitude=slice(39.25, 29.75), longitude=slice(-105.25, -94.75))

uvq_new = uvq_new.rename({"valid_time": "time", "pressure_level": "level"})

uvq_new["time"] = uvq_new["time"] - pd.Timedelta(hours=6)

uvq_new = uvq_new.where(
    
    (uvq_new.time.dt.month.isin([5, 6, 7, 8, 9])) &
    
    (uvq_new.time.dt.year.isin(range(2001, 2026))),  # 2022–2025 will survive
    
    drop=True)

uvq_new = uvq_new.sortby("level", ascending=True)

## combine yizhou & my data

# -------------------------------------------------
# 1) Keep only shared variables
# -------------------------------------------------

common_vars = sorted(set(uvq_yz.data_vars) & set(uvq_new.data_vars))

uvq_yz = uvq_yz[common_vars]

uvq_new = uvq_new[common_vars]

uvq_yz = uvq_yz.drop_vars("expver", errors="ignore")

uvq_new = uvq_new.drop_vars("expver", errors="ignore")

# -------------------------------------------------
# 2) Concatenate along time
# -------------------------------------------------

uvq = xr.concat([uvq_yz, uvq_new], dim="time", join="exact")   # forces identical lat/lon/level grids

# -------------------------------------------------
# 3) Sort by time
# -------------------------------------------------

uvq = uvq.sortby("time")

# -------------------------------------------------
# 4) Drop duplicate timestamps (overlap years)
# -------------------------------------------------

t = pd.Index(uvq.time.values)

dupes = t.duplicated(keep="first")

if dupes.any():
    
    uvq = uvq.isel(
        
        time=~xr.DataArray(dupes, dims="time"))
    
print("uvq ready")

# Define datasets

_0lst = uvq.sel(time=uvq.time.dt.hour == 0)

_3lst = uvq.sel(time=uvq.time.dt.hour == 3)

_6lst = uvq.sel(time=uvq.time.dt.hour == 6)

_9lst = uvq.sel(time=uvq.time.dt.hour == 9)

_12lst = uvq.sel(time=uvq.time.dt.hour == 12)

_15lst = uvq.sel(time=uvq.time.dt.hour == 15)

_18lst = uvq.sel(time=uvq.time.dt.hour == 18)

_21lst = uvq.sel(time=uvq.time.dt.hour == 21)

datasets = {'0lst': _0lst, '3lst': _3lst, '6lst': _6lst, '9lst': _9lst, '12lst': _12lst, '15lst': _15lst, '18lst': _18lst, 
            '21lst': _21lst}

# Load master DataFrame

master_path = os.path.join(home_dir, "master_w_sp_2001_2025.pkl")

with open(master_path, 'rb') as f:
    
    master = pickle.load(f)

print("Master loaded")

# load surface pressure

with open(os.path.join(home_dir, 'pressure.pkl'), 'rb') as f:
    
    pressure = pickle.load(f) * 100  # Pa